<a href="https://colab.research.google.com/github/IvanMorsin/Forecasting-electrical-power-in-multi-storey-residential-buildings/blob/main/Notebook_1_(%D0%97%D0%B0%D0%B3%D1%80%D1%83%D0%B7%D0%BA%D0%B0_%D0%B4%D0%B0%D0%BD%D0%BD%D1%8B%D1%85).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
files = {
    1: "House1.xlsx",
    2: "House2.xlsx",
    3: "House3.xlsx",
    4: "House4.xlsx",
    5: "House5.xlsx",
    6: "House6.xlsx",
    7: "House7.xlsx",
    8: "House8.xlsx",
}

all_houses = []

for house_id, filename in files.items():

    df_raw = pd.read_excel(filename)
    block_starts = [0, 7, 14]
    blocks = []

    for start in block_starts:
        block = df_raw.iloc[:, [start, start+1, start+3]].copy()
        block.columns = ["date", "time", "power_kw"]
        block = block.dropna(subset=["date", "power_kw"])
        block = block[block["power_kw"] > 0]
        blocks.append(block)

    df = pd.concat(blocks, ignore_index=True)

    # Исправление 24:00 на 00:00 следующего дня
    df["datetime_str"] = df["date"].astype(str) + " " + df["time"].astype(str)
    mask_24 = df["datetime_str"].str.contains(" 24:")
    df.loc[mask_24, "datetime_str"] = df.loc[mask_24, "datetime_str"].str.replace(
        " 24:", " 00:", regex=False
    )
    df["timestamp"] = pd.to_datetime(df["datetime_str"], format="mixed")
    df.loc[mask_24, "timestamp"] = df.loc[mask_24, "timestamp"] + pd.Timedelta(days=1)

    df = df[["timestamp", "power_kw"]].sort_values("timestamp").reset_index(drop=True)
    df = df.drop_duplicates(subset="timestamp")
    df["house_id"] = house_id

    all_houses.append(df)
    print(f"строк: {len(df)}, период: {df['timestamp'].min()} - {df['timestamp'].max()}")

строк: 37486, период: 2017-11-11 00:30:00 - 2020-01-01 00:00:00
строк: 37486, период: 2017-11-11 00:30:00 - 2020-01-01 00:00:00
строк: 37397, период: 2017-11-11 00:30:00 - 2020-01-01 00:00:00
строк: 51888, период: 2017-01-01 00:30:00 - 2020-01-01 00:00:00
строк: 52560, период: 2017-01-01 00:30:00 - 2020-01-01 00:00:00
строк: 50498, период: 2017-01-01 00:30:00 - 2020-01-01 00:00:00
строк: 50051, период: 2017-01-01 00:30:00 - 2020-01-01 00:00:00
строк: 51326, период: 2017-01-01 00:30:00 - 2019-12-06 07:00:00


In [ ]:
for df in all_houses:
    house_id = df["house_id"].iloc[0]

    full_index = pd.date_range(
        start=df["timestamp"].min(),
        end=df["timestamp"].max(),
        freq="30min"
    )

    # пропуски
    n_missing = len(full_index) - len(df)
    pct = n_missing / len(full_index) * 100

    print(f"Дом {house_id}:")
    print(f"Количество ожидаемых значений: {len(full_index)}")
    print(f"Количество фактических значений: {len(df)}")
    print(f"Количество пропусков: {n_missing} ({round(pct, 2)}%)")
    print()

Дом 1:
Количество ожидаемых значений: 37488
Количество фактических значений: 37486
Количество пропусков: 2 (0.01%)

Дом 2:
Количество ожидаемых значений: 37488
Количество фактических значений: 37486
Количество пропусков: 2 (0.01%)

Дом 3:
Количество ожидаемых значений: 37488
Количество фактических значений: 37397
Количество пропусков: 91 (0.24%)

Дом 4:
Количество ожидаемых значений: 52560
Количество фактических значений: 51888
Количество пропусков: 672 (1.28%)

Дом 5:
Количество ожидаемых значений: 52560
Количество фактических значений: 52560
Количество пропусков: 0 (0.0%)

Дом 6:
Количество ожидаемых значений: 52560
Количество фактических значений: 50498
Количество пропусков: 2062 (3.92%)

Дом 7:
Количество ожидаемых значений: 52560
Количество фактических значений: 50051
Количество пропусков: 2509 (4.77%)

Дом 8:
Количество ожидаемых значений: 51326
Количество фактических значений: 51326
Количество пропусков: 0 (0.0%)



In [ ]:
global_start = min(df["timestamp"].min() for df in all_houses)
global_end   = max(df["timestamp"].max() for df in all_houses)
full_index   = pd.date_range(start=global_start, end=global_end, freq="30min")

print(f"Общий период: {global_start} — {global_end}")
print(f"Всего точек в индексе: {len(full_index)}")

# сведение в широкий датафрейм
df_wide = pd.DataFrame({"timestamp": full_index})

for df in all_houses:
    house_id = df["house_id"].iloc[0]
    col_name = f"house_{house_id}"

    # джойн по timestamp
    df_wide = df_wide.merge(
        df[["timestamp", "power_kw"]].rename(columns={"power_kw": col_name}),
        on="timestamp",
        how="left"
    )

Общий период: 2017-01-01 00:30:00 — 2020-01-01 00:00:00
Всего точек в индексе: 52560


In [ ]:
print(df_wide.shape)

(52560, 9)


In [ ]:
print(df_wide.head())

            timestamp  house_1  house_2  house_3  house_4  house_5  house_6  \
0 2017-01-01 00:30:00      NaN      NaN      NaN    50.22   29.765    79.86   
1 2017-01-01 01:00:00      NaN      NaN      NaN    46.74   26.845    71.94   
2 2017-01-01 01:30:00      NaN      NaN      NaN    44.42   25.150    69.84   
3 2017-01-01 02:00:00      NaN      NaN      NaN    44.48   25.005    65.46   
4 2017-01-01 02:30:00      NaN      NaN      NaN    40.44   23.535    59.82   

   house_7  house_8  
0    94.35    43.25  
1    88.63    43.45  
2    83.33    41.10  
3    80.59    38.65  
4    71.40    36.40  


In [ ]:
df_wide.to_csv("df_raw.csv", index=False)